# 01 — PDF Extraction + Quality Assessment
Ekstraksi teks dari PDF publikasi BPS menggunakan PyMuPDF + pdfplumber.
Disesuaikan dengan Panduan Pembuatan Publikasi BPS Edisi 2023.

In [1]:
!pip install PyMuPDF pdfplumber PySastrawi -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 76.4 MB/s eta 0:00:00


In [2]:
import fitz
import pdfplumber
import json
import re
from pathlib import Path
from collections import Counter
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from IPython.display import display
import pandas as pd

## 1. Konfigurasi

In [3]:
INPUT_DIR  = Path("/kaggle/input/datasets/muhammadnabilul/bps-publications")
df_meta    = pd.read_excel(INPUT_DIR / 'List_publikasi.xlsx')
PDF_DIR    = INPUT_DIR / "pdf"
OUT_DIR    = Path("/kaggle/working/extracted")
OUT_DIR.mkdir(exist_ok=True)

# Threshold quality
MIN_CHARS_PER_PAGE   = 100
MIN_COVERAGE_RATIO   = 0.70
MAX_NONASCII_RATIO   = 0.05
MIN_WORD_VALID_RATIO = 0.60
REPETITION_THRESHOLD = 0.40

# Threshold deteksi dwibahasa:
# Kalau blok teks di kolom kanan > N, halaman dianggap dwibahasa
BILINGUAL_MIN_RIGHT_BLOCKS = 2

## 2. Konstanta Struktur Publikasi BPS
Berdasarkan Panduan Pembuatan Publikasi BPS Edisi 2023, Bab 4.

In [4]:
# Regex nomor halaman romawi (halaman pendahuluan)
ROMAN_PAGE = re.compile(
    r'^(i{1,3}|iv|vi{0,3}|ix|x{1,3}|xi{0,3}|xiv|xv|xvi|xvii|xviii|xix|xx)$',
    re.IGNORECASE
)

# Regex nomor halaman arab
ARABIC_PAGE = re.compile(r'^\d{1,4}$')

# Judul halaman pendahuluan yang harus di-skip
# Hanya yang benar-benar tidak informatif
SKIP_PRELIMINARY_TITLES = {
    'tim penyusun', 'penyusun', 'editorial board',
    'kata sambutan', 'sambutan',
    'kata pengantar', 'preface', 'foreword',
    'daftar pustaka', 'references', 'bibliography',
}

# Halaman yang DISIMPAN
KEEP_PRELIMINARY_TITLES = {
    # Eksplisit di batasan penelitian
    'daftar isi', 'table of contents', 'contents',
    'daftar tabel', 'list of tables',
    'daftar gambar', 'list of figures', 'list of charts',
    'daftar grafik', 'daftar diagram',
    # Konten naratif penting
    'abstraksi', 'abstract', 'ringkasan', 'ringkasan eksekutif',
    'executive summary', 'summary',
    'penjelasan teknis', 'technical notes',
    'penjelasan umum', 'general notes',
    'metodologi', 'methodology',
}

print('Konstanta struktur BPS dimuat.')

Konstanta struktur BPS dimuat.


## 3. Fungsi Deteksi Zona Halaman
Halaman pendahuluan (romawi) vs halaman isi (arab) sesuai panduan BPS.

In [5]:
def detect_page_zone(text):
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    for line in lines[:5] + lines[-3:]:
        if ROMAN_PAGE.match(line):
            return 'preliminary'
        if ARABIC_PAGE.match(line):
            return 'content'
    return 'unknown'

def should_skip_preliminary(text):
    lines_lower = [l.strip().lower() for l in text.split('\n') if l.strip()]
    first_lines = ' '.join(lines_lower[:10])
    # Simpan kalau ada judul yang penting
    if any(t in first_lines for t in KEEP_PRELIMINARY_TITLES):
        return False
    # Skip kalau ada judul yang tidak perlu
    if any(t in first_lines for t in SKIP_PRELIMINARY_TITLES):
        return True
    return False

## 4. Fungsi Deteksi & Ekstraksi Dwibahasa
Sesuai panduan BPS: publikasi dwibahasa menggunakan format dua kolom,
Bahasa Indonesia di kolom kiri, bahasa asing di kolom kanan.

In [6]:
def is_bilingual_page(page):
    page_width = page.rect.width
    blocks = page.get_text('blocks', sort=True)
    right_blocks = [
        b for b in blocks
        if b[0] > page_width * 0.50
        and len(b[4].strip()) > 20
    ]
    return len(right_blocks) >= BILINGUAL_MIN_RIGHT_BLOCKS

def extract_left_column(page):
    page_width = page.rect.width
    blocks = page.get_text('blocks', sort=True)
    left_blocks = [
        b[4] for b in blocks
        if b[0] < page_width * 0.55
        and b[4].strip()
    ]
    return '\n'.join(left_blocks)

## 5. Fungsi Deteksi Header/Footer Berulang
Running title (judul buku di halaman genap, judul bab di halaman ganjil)
sesuai panduan BPS akan terdeteksi dan dihapus.

In [7]:
def detect_repeated_lines(pages_text, threshold=REPETITION_THRESHOLD):
    first_lines, last_lines = [], []
    for text in pages_text:
        lines = [l.strip() for l in text.split('\n') if l.strip()]
        if lines:
            first_lines.append(lines[0])
            last_lines.append(lines[-1])

    n = max(len(pages_text), 1)
    repeated = set()

    for line, cnt in Counter(first_lines).most_common(20):
        if cnt / n >= threshold and len(line) > 3:
            repeated.add(line)
    for line, cnt in Counter(last_lines).most_common(20):
        if cnt / n >= threshold and len(line) > 3:
            repeated.add(line)

    # Tambah URL BPS yang selalu muncul
    repeated.add('https://www.bps.go.id')
    repeated.add('http://www.bps.go.id')

    return repeated

## 6. Fungsi Quality Metrics

In [8]:
factory   = StopWordRemoverFactory()
stopwords = set(factory.get_stop_words())

INDO_COMMON = {
    'dan','yang','di','ini','itu','dengan','untuk','dari','pada','adalah',
    'dalam','tidak','sebagai','oleh','juga','telah','akan','atau','tahun',
    'data','jumlah','persentase','penduduk','indonesia','provinsi','kabupaten',
    'nasional','total','rata','nilai','tingkat','persen','angka','statistik',
    'tabel','gambar','bab','subbab','halaman','sumber','catatan','keterangan',
    'hasil','survei','sensus','bps','pusat','daerah','kota','wilayah',
} | stopwords

def word_validity_ratio(text):
    words = re.findall(r'[a-zA-Z]+', text.lower())
    if not words:
        return 1.0
    valid = sum(1 for w in words if w in INDO_COMMON or len(w) >= 3)
    return valid / len(words)

def non_ascii_ratio(text):
    if not text:
        return 0.0
    non_ascii = sum(1 for c in text if ord(c) > 127)
    return non_ascii / len(text)

def assess_page(text):
    n_chars = len(text.strip())
    return {
        'n_chars'         : n_chars,
        'is_empty'        : n_chars < MIN_CHARS_PER_PAGE,
        'non_ascii_ratio' : round(non_ascii_ratio(text), 4),
        'word_valid_ratio': round(word_validity_ratio(text), 4),
    }

## 7. Fungsi Utama Ekstraksi Per Dokumen

In [9]:
def extract_document(pdf_path, meta):
    doc_fitz  = fitz.open(str(pdf_path))
    total_hal = len(doc_fitz)

    # Pass 1: ekstrak semua teks + deteksi dwibahasa per halaman
    raw_pages   = []
    zone_pages  = []
    bilingual_count = 0

    for page in doc_fitz:
        if is_bilingual_page(page):
            text = extract_left_column(page)
            bilingual_count += 1
        else:
            text = page.get_text()
        raw_pages.append(text)
        zone_pages.append(detect_page_zone(text))

    doc_fitz.close()
    is_bilingual_doc = bilingual_count > total_hal * 0.2

    # Deteksi header/footer berulang (running title BPS)
    repeated_lines = detect_repeated_lines(raw_pages)

    # Pass 2: ekstrak tabel via pdfplumber
    tables_by_page = {}
    with pdfplumber.open(str(pdf_path)) as pdf:
        for i, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            if tables:
                tables_by_page[i] = tables

    # Pass 3: proses per halaman
    pages_out   = []
    flagged     = []
    n_extracted = 0
    n_skipped_preliminary = 0

    for i, (raw_text, zone) in enumerate(zip(raw_pages, zone_pages)):
        # Hapus header/footer berulang
        lines = raw_text.split('\n')
        lines = [l for l in lines if l.strip() not in repeated_lines]
        text  = '\n'.join(lines).strip()

        # Skip halaman pendahuluan yang tidak informatif
        if zone == 'preliminary' and should_skip_preliminary(text):
            n_skipped_preliminary += 1
            continue

        # Skip halaman kosong
        if len(text.strip()) < MIN_CHARS_PER_PAGE:
            continue

        # Tambahkan konten tabel
        if i in tables_by_page:
            for table in tables_by_page[i]:
                for row in table:
                    row_text = ' | '.join(
                        str(cell).strip() for cell in row if cell
                    )
                    if row_text:
                        text += '\n' + row_text

        metrics = assess_page(text)
        issues  = []

        if metrics['non_ascii_ratio'] > MAX_NONASCII_RATIO:
            issues.append('non_ascii_high')
        if metrics['word_valid_ratio'] < MIN_WORD_VALID_RATIO:
            issues.append('word_validity_low')

        if issues:
            flagged.append({
                'halaman': i + 1,
                'zone'   : zone,
                'issues' : issues,
                **metrics
            })

        n_extracted += 1
        pages_out.append({
            'chunk_id': f"{meta['nama_file'].replace('.pdf','')}__p{i+1:04d}",
            'halaman' : i + 1,
            'zone'    : zone,
            'teks'    : text,
            'metrics' : metrics,
        })

    # Quality score
    usable_pages = total_hal - n_skipped_preliminary
    coverage     = n_extracted / usable_pages if usable_pages else 0
    avg_chars    = sum(p['metrics']['n_chars'] for p in pages_out) / max(n_extracted, 1)
    avg_valid    = sum(p['metrics']['word_valid_ratio'] for p in pages_out) / max(n_extracted, 1)
    avg_nonasci  = sum(p['metrics']['non_ascii_ratio'] for p in pages_out) / max(n_extracted, 1)

    if coverage >= MIN_COVERAGE_RATIO and avg_valid >= MIN_WORD_VALID_RATIO:
        recommendation = 'accept'
    elif coverage >= 0.50:
        recommendation = 'review'
    else:
        recommendation = 'poor'

    quality = {
        'total_halaman'          : total_hal,
        'halaman_preliminary_skip': n_skipped_preliminary,
        'halaman_diekstrak'      : n_extracted,
        'coverage_ratio'         : round(coverage, 4),
        'is_bilingual'           : is_bilingual_doc,
        'bilingual_pages'        : bilingual_count,
        'avg_chars_per_page'     : round(avg_chars, 1),
        'avg_word_valid'         : round(avg_valid, 4),
        'avg_non_ascii'          : round(avg_nonasci, 4),
        'repeated_lines'         : list(repeated_lines),
        'flagged_pages'          : flagged,
        'n_flagged'              : len(flagged),
        'overall_quality'        : 'good' if recommendation == 'accept' else 'warning' if recommendation == 'review' else 'poor',
        'recommendation'         : recommendation,
    }

    return pages_out, quality

## 8. Load Metadata & Jalankan Ekstraksi & Simpan

In [10]:
df_meta.columns = ['tahun', 'judul', 'bps_url', 'nama_file']
df_meta['tahun'] = df_meta['tahun'].astype(str)
metadata_list = df_meta.to_dict(orient='records')

print(f'Total dokumen: {len(df_meta)}')

Total dokumen: 47


In [11]:
all_results     = []
quality_reports = []
total           = len(df_meta)

for idx, row in df_meta.iterrows():
    tahun     = str(row['tahun'])
    nama_file = row['nama_file']
    judul     = row['judul']
    pdf_path  = PDF_DIR / tahun / nama_file

    print(f'[{idx+1}/{total}] {judul[:60]}')

    if not pdf_path.exists():
        print(f'  ✗ File tidak ditemukan: {pdf_path}')
        qr = {
            'nama_file': nama_file, 'judul': judul, 'tahun': tahun,
            'recommendation': 'file-missing', 'overall_quality': 'poor',
        }
        quality_reports.append(qr)
        continue

    pages, quality = extract_document(pdf_path, row)
    quality.update({'nama_file': nama_file, 'judul': judul, 'tahun': tahun})
    quality_reports.append(quality)

    doc_result = {
        'pub_id'   : row.get('pub_id', ''),
        'judul'    : judul,
        'tahun'    : tahun,
        'nama_file': nama_file,
        'bps_url'  : row.get('bps_url', ''),
        'n_halaman': quality['total_halaman'],
        'n_chunks' : len(pages),
        'bilingual': quality['is_bilingual'],
        'quality'  : quality['overall_quality'],
        'pages'    : pages,
    }
    all_results.append(doc_result)

    # Simpan langsung ke disk — tidak tunggu semua selesai
    fname = nama_file.replace('.pdf', '.json')
    with open(OUT_DIR / fname, 'w', encoding='utf-8') as f:
        json.dump(doc_result, f, ensure_ascii=False, indent=2)

    # Update quality report setiap iterasi
    with open(OUT_DIR / 'quality_report.json', 'w', encoding='utf-8') as f:
        json.dump(quality_reports, f, ensure_ascii=False, indent=2)

    bi = ' [dwibahasa]' if quality['is_bilingual'] else ''
    print(f'  ✓ {quality["halaman_diekstrak"]}/{quality["total_halaman"]} hal'
          f' | skip_prelim={quality["halaman_preliminary_skip"]}'
          f' | {quality["overall_quality"]}{bi}')
    if quality['flagged_pages']:
        print(f'  ⚑ {quality["n_flagged"]} halaman di-flag')

print(f'\nSelesai: {len(all_results)} dokumen')

[1/47] Statistik Indonesia 2023
  ✓ 796/830 hal | skip_prelim=0 | good [dwibahasa]
  ⚑ 15 halaman di-flag
[2/47] Indeks Pembangunan Manusia 2022
  ✓ 154/180 hal | skip_prelim=0 | good [dwibahasa]
[3/47] Indeks Perilaku Anti Korupsi 2023
  ✓ 151/184 hal | skip_prelim=2 | good
[4/47] Indikator Kesejahteraan Rakyat 2023
  ✓ 200/230 hal | skip_prelim=2 | good [dwibahasa]
[5/47] Kesejahteraan Anak Indonesia: Analisis Deprivasi Hak Anak Mu
  ✓ 86/98 hal | skip_prelim=1 | good
[6/47] Kesejahteraan Anak Indonesia: Analisis Kemiskinan Anak Monet
  ✓ 92/105 hal | skip_prelim=3 | good [dwibahasa]
  ⚑ 2 halaman di-flag
[7/47] Laporan Indeks Khusus Penanganan Stunting 2021–2022
  ✓ 94/118 hal | skip_prelim=6 | good
  ⚑ 2 halaman di-flag
[8/47] Laporan Indeks Khusus Penanganan Stunting Kabupaten/Kota 202
  ✓ 606/714 hal | skip_prelim=10 | good
  ⚑ 1 halaman di-flag
[9/47] Profil Anak Usia Dini 2023
  ✓ 166/236 hal | skip_prelim=2 | good
[10/47] Profil Statistik Kesehatan 2023
  ✓ 447/484 hal | skip_

## 9. Ringkasan Quality Report

In [12]:
df_q = pd.DataFrame(quality_reports)

print('=== RINGKASAN ===')
print(df_q['recommendation'].value_counts().to_string())
print(f'\nRata coverage  : {df_q["coverage_ratio"].mean():.3f}')
print(f'Rata word valid: {df_q["avg_word_valid"].mean():.3f}')
print(f'Dok dwibahasa  : {df_q["is_bilingual"].sum()}')

perlu_review = df_q[df_q['recommendation'].isin(['review','poor','file-missing'])]
if len(perlu_review):
    print('\n--- Perlu Review Manual ---')
    display(perlu_review[['judul','tahun','coverage_ratio','avg_word_valid','is_bilingual','recommendation']])
else:
    print('\n✓ Semua dokumen OK')

=== RINGKASAN ===
recommendation
accept    44
review     3

Rata coverage  : 0.835
Rata word valid: 0.972
Dok dwibahasa  : 13

--- Perlu Review Manual ---


,judul,tahun,coverage_ratio,avg_word_valid,is_bilingual,recommendation
36,Infografis Statistik Kesejahteraan Rakyat 2025,2025,0.5000,0.9796,False,review
37,Profil Anak Usia Dini 2025,2025,0.6927,0.9890,False,review
44,Statistik Penunjang Pendidikan 2024,2025,0.6935,0.9799,False,review


## 10. Inspeksi Detail per Dokumen

In [13]:
# # Ganti sesuai file yang ingin dicek
# TARGET_FILE = 'statistik-kesehatan-2022.json'
# TARGET_PAGE = 31

# target_path = OUT_DIR / TARGET_FILE
# if target_path.exists():
#     with open(target_path, encoding='utf-8') as f:
#         doc = json.load(f)

#     qr = next((q for q in quality_reports
#                if q.get('nama_file') == TARGET_FILE.replace('.json','.pdf')), {})

#     print(f'Dokumen   : {doc["judul"]}')
#     print(f'Kualitas  : {qr.get("overall_quality")} | {qr.get("recommendation")}')
#     print(f'Coverage  : {qr.get("coverage_ratio")}')
#     print(f'Dwibahasa : {qr.get("is_bilingual")} ({qr.get("bilingual_pages")} halaman)')
#     print(f'Skip prelim: {qr.get("halaman_preliminary_skip")} halaman')
#     print(f'Flagged   : {qr.get("n_flagged")} halaman')
#     print(f'Repeated  : {qr.get("repeated_lines")}')

#     page = next((p for p in doc['pages'] if p['halaman'] == TARGET_PAGE), None)
#     if page:
#         print(f'\n--- Hal {TARGET_PAGE} | zone={page["zone"]} | {page["metrics"]["n_chars"]} chars ---')
#         print(page['teks'][:800])
#     else:
#         print(f'Halaman {TARGET_PAGE} tidak ditemukan atau di-skip')
# else:
#     print(f'File {TARGET_FILE} tidak ditemukan')